In [199]:
#%cd ../../

In [ ]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
import uuid
import optuna
from sklearn.metrics import roc_auc_score, log_loss
from helpers import (
    load_train_data,
    load_test_data,
)
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [201]:
train = load_train_data()
print(train.shape)

(594194, 21)


In [202]:
test = load_test_data()
print(test.shape)

(254655, 20)


In [203]:
train = train.iloc[:, 1:] # Remove id
train.SeniorCitizen = train.SeniorCitizen.map({0: 'No', 1: 'Yes'})
train.Churn = train.Churn.map({'Yes': 1, 'No': 0}) # Label encode
print(train.shape)

(594194, 20)


In [204]:
# Num and cat cols
cat_cols = train.select_dtypes(['object']).columns.to_list()
num_cols = train.select_dtypes(['int', 'float']).columns.to_list()
for col in cat_cols:
    train[col] = train[col].astype('category')
#print(cat_cols)
print(len(cat_cols))
#print(num_cols)
print(len(num_cols))

16
4


In [205]:
test = test.iloc[:, 1:] # Remove id
test.SeniorCitizen = test.SeniorCitizen.map({0: 'No', 1: 'Yes'})
print(test.shape)

(254655, 19)


In [206]:
# Num and cat cols
cat_cols = test.select_dtypes(['object']).columns.to_list()
num_cols = test.select_dtypes(['int', 'float']).columns.to_list()
for col in cat_cols:
    test[col] = test[col].astype('category')
#print(cat_cols)
print(len(cat_cols))
#print(num_cols)
print(len(num_cols))

16
3


In [207]:
# Split X and y
X = train.drop('Churn', axis=1)
y = train.Churn

In [208]:
def objective(trial):

    params = {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "num_leaves": trial.suggest_int("num_leaves", 16, 64),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "seed": 42,
        "verbosity": -1,
        "n_jobs": -1
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    oof_preds = np.zeros(len(X))

    train_auc_scores = []
    val_auc_scores = []

    for train_idx, val_idx in skf.split(X, y):

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        train_data = lgb.Dataset(X_train, label=y_train)
        val_data = lgb.Dataset(X_val, label=y_val)

        model = lgb.train(
            params,
            train_data,
            num_boost_round=100,
            valid_sets=[val_data],
            callbacks=[lgb.early_stopping(10)]
        )

        train_preds = model.predict(X_train)
        val_preds = model.predict(X_val)

        train_auc = roc_auc_score(y_train, train_preds)
        val_auc = roc_auc_score(y_val, val_preds)

        train_auc_scores.append(train_auc)
        val_auc_scores.append(val_auc)

        oof_preds[val_idx] = val_preds

    oof_auc = roc_auc_score(y, oof_preds)

    # ------------------------
    # Logging useful attributes
    # ------------------------

    trial.set_user_attr("oof_auc", oof_auc) # AUC using combined out-of-fold predictions
    trial.set_user_attr("train_auc_mean", np.mean(train_auc_scores)) # Average AUC on training data across all folds
    trial.set_user_attr("val_auc_mean", np.mean(val_auc_scores)) # Average validation AUC across folds
    trial.set_user_attr("val_auc_std", np.std(val_auc_scores)) # Standard deviation of validation AUC across folds
    trial.set_user_attr("model", "lightgbm")

    return oof_auc

In [209]:
study = optuna.create_study(
    direction="maximize",
    study_name="lgbm_tuning_v1",
    storage="sqlite:///optuna.db",
    load_if_exists=True
)

study.optimize(objective, n_trials=5, show_progress_bar=True, n_jobs=-1)


print("Best AUC:", study.best_value)
print("Best params:", study.best_params)

[I 2026-03-07 02:17:52,958] A new study created in RDB with name: lgbm_tuning_v1
  0%|          | 0/5 [00:00<?, ?it/s]

Training until validation scores don't improve for 10 rounds
Training until validation scores don't improve for 10 rounds
Training until validation scores don't improve for 10 rounds
Training until validation scores don't improve for 10 rounds
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.906174
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.907401
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.913892
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.910684
Training until validation scores don't improve for 10 rounds
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.911959
Did not meet early stopping. Best iteration is:
[100]	valid_0

Best trial: 0. Best value: 0.906641:  20%|██        | 1/5 [00:25<01:42, 25.57s/it]

[I 2026-03-07 02:18:18,496] Trial 0 finished with value: 0.9066413403210675 and parameters: {'learning_rate': 0.019977183134731173, 'num_leaves': 59, 'max_depth': 3, 'subsample': 0.9095114542418473, 'colsample_bytree': 0.9543694958874129}. Best is trial 0 with value: 0.9066413403210675.
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.911266
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.914286
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.912742
Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.912368
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.912348
Training until validation scores don't improve for 10 rounds
Training until va

Best trial: 3. Best value: 0.91109:  40%|████      | 2/5 [00:41<00:59, 19.75s/it] 

...

[I 2026-03-07 02:18:38,743] Trial 2 finished with value: 0.9142333540807656 and parameters: {'learning_rate': 0.07586709726296281, 'num_leaves': 46, 'max_depth': 5, 'subsample': 0.7127314246363239, 'colsample_bytree': 0.9490531112578929}. Best is trial 2 with value: 0.9142333540807656.
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.910994


Best trial: 2. Best value: 0.914233:  80%|████████  | 4/5 [00:48<00:08,  8.65s/it]

[I 2026-03-07 02:18:41,024] Trial 1 finished with value: 0.9125873462453427 and parameters: {'learning_rate': 0.035305303121810246, 'num_leaves': 39, 'max_depth': 5, 'subsample': 0.9647678746392933, 'colsample_bytree': 0.6173009538804425}. Best is trial 2 with value: 0.9142333540807656.
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.910597


Best trial: 2. Best value: 0.914233: 100%|██████████| 5/5 [00:49<00:00,  9.94s/it]

[I 2026-03-07 02:18:42,619] Trial 4 finished with value: 0.9122519112103629 and parameters: {'learning_rate': 0.024880739318510932, 'num_leaves': 39, 'max_depth': 6, 'subsample': 0.9264104361032611, 'colsample_bytree': 0.851031337278152}. Best is trial 2 with value: 0.9142333540807656.
Best AUC: 0.9142333540807656
Best params: {'learning_rate': 0.07586709726296281, 'num_leaves': 46, 'max_depth': 5, 'subsample': 0.7127314246363239, 'colsample_bytree': 0.9490531112578929}


In [210]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_learning_rate,params_max_depth,params_num_leaves,params_subsample,user_attrs_model,user_attrs_oof_auc,user_attrs_train_auc_mean,user_attrs_val_auc_mean,user_attrs_val_auc_std,state
0,0,0.906641,2026-03-07 02:17:52.975323,2026-03-07 02:18:18.468876,0 days 00:00:25.493553,0.954369,0.019977,3,59,0.909511,lightgbm,0.906641,0.906814,0.906710,0.000984,COMPLETE
1,1,0.912587,2026-03-07 02:17:52.980862,2026-03-07 02:18:40.960519,0 days 00:00:47.979657,0.617301,0.035305,5,39,0.964768,lightgbm,0.912587,0.913117,0.912606,0.000977,COMPLETE
2,2,0.914233,2026-03-07 02:17:53.003498,2026-03-07 02:18:38.716628,0 days 00:00:45.713130,0.949053,0.075867,5,46,0.712731,lightgbm,0.914233,0.915289,0.914250,0.000951,COMPLETE
3,3,0.911090,2026-03-07 02:17:53.044952,2026-03-07 02:18:34.132580,0 days 00:00:41.087628,0.928108,0.022961,6,18,0.788819,lightgbm,0.911090,0.911432,0.911136,0.001015,COMPLETE
4,4,0.912252,2026-03-07 02:17:53.058930,2026-03-07 02:18:42.601777,0 days 00:00:49.542847,0.851031,0.024881,6,39,0.926410,lightgbm,0.912252,0.912863,0.912295,0.001003,COMPLETE


In [211]:
for trial in study.trials:
    print(trial.number, trial.value, trial.user_attrs)

0 0.9066413403210675 {'model': 'lightgbm', 'oof_auc': 0.9066413403210675, 'train_auc_mean': 0.9068136191358043, 'val_auc_mean': 0.9067096469680742, 'val_auc_std': 0.0009841810835938942}
1 0.9125873462453427 {'model': 'lightgbm', 'oof_auc': 0.9125873462453427, 'train_auc_mean': 0.9131172432450642, 'val_auc_mean': 0.9126056226605952, 'val_auc_std': 0.0009773496701027213}
2 0.9142333540807656 {'model': 'lightgbm', 'oof_auc': 0.9142333540807656, 'train_auc_mean': 0.9152894191626686, 'val_auc_mean': 0.9142498794043556, 'val_auc_std': 0.0009508786919302555}
3 0.91108963813894 {'model': 'lightgbm', 'oof_auc': 0.91108963813894, 'train_auc_mean': 0.91143186497358, 'val_auc_mean': 0.9111363445577831, 'val_auc_std': 0.0010152550715241619}
4 0.9122519112103629 {'model': 'lightgbm', 'oof_auc': 0.9122519112103629, 'train_auc_mean': 0.9128629581462426, 'val_auc_mean': 0.9122952403280123, 'val_auc_std': 0.0010030737471647642}


In [212]:
study.trials

[FrozenTrial(number=0, state=<TrialState.COMPLETE: 1>, values=[0.9066413403210675], datetime_start=datetime.datetime(2026, 3, 7, 2, 17, 52, 975323), datetime_complete=datetime.datetime(2026, 3, 7, 2, 18, 18, 468876), params={'learning_rate': 0.019977183134731173, 'num_leaves': 59, 'max_depth': 3, 'subsample': 0.9095114542418473, 'colsample_bytree': 0.9543694958874129}, user_attrs={'model': 'lightgbm', 'oof_auc': 0.9066413403210675, 'train_auc_mean': 0.9068136191358043, 'val_auc_mean': 0.9067096469680742, 'val_auc_std': 0.0009841810835938942}, system_attrs={}, intermediate_values={}, distributions={'learning_rate': FloatDistribution(high=0.1, log=False, low=0.01, step=None), 'num_leaves': IntDistribution(high=64, log=False, low=16, step=1), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'subsample': FloatDistribution(high=1.0, log=False, low=0.6, step=None), 'colsample_bytree': FloatDistribution(high=1.0, log=False, low=0.6, step=None)}, trial_id=6, value=None),
 Froze